# StashFace — الخطوة 2: الفحص والمقارنة (Query)

**مفيش داعي GPU خالص هنا** - المرحلة دي بس بتقارن embeddings جاهزة (numpy عادي)، مش بتشغل أي موديل. تقدر تسيب الـAccelerator على None وتوفر حصة الـGPU بتاعتك.

شرط إنك تكون خلصت `stashface_build.ipynb` الأول ورفع القاعدة (`face_db.npz`) على HF.

In [ ]:
# ============================================================
# خلية الإعداد — عدّل القيم دي بس، والباقي متلمسوش
# ============================================================

# رابط الـGitHub repo بتاع المشروع
GITHUB_REPO_URL = "https://github.com/kareemkamal10/stashface_pipeline"

# الـtoken بتاع حسابك على Hugging Face (لازم يكون عنده صلاحية Write)
HF_TOKEN = " "

# نفس الـdataset اللي notebook البناء رفع عليه القاعدة
HF_DATASET_ID = "abdelwahabnabil500/datafile"

In [ ]:
# ============================================================
# تحميل الكود وتثبيت المكتبات - خفيفة جدًا هنا، مفيش insightface/onnxruntime
# ============================================================
import os

os.environ["HF_HUB_DISABLE_UPDATE_CHECK"] = "1"

# 1) تحميل الكود من الـGitHub repo
!git clone {GITHUB_REPO_URL} /kaggle/temp/stashface_pipeline
%cd /kaggle/temp/stashface_pipeline

# 2) المكتبات المطلوبة للمقارنة بس (numpy وrequests وPillow غالبًا موجودين
# أصلاً في صورة Kaggle الأساسية)
!pip install -q -U "huggingface_hub[cli]>=1.13.0"

# 3) تسجيل الدخول لـHugging Face بالـtoken بتاعك
from huggingface_hub import login
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

## المقارنة

بتحمّل `face_db.npz` من HF (اللي `stashface_build.ipynb` رفعها)، وتعمل المقارنة (top-10 لكل عنصر + تجميع union-find)، وترفع `duplicate_groups.json` على `reports/` جوا نفس الـdataset.

دي هتاخد دقايق بس، مش ساعات.

In [ ]:
!python dedupe_pipeline.py --mode query --hf-dataset-id {HF_DATASET_ID} --hf-token {HF_TOKEN}

## ملخص النتايج

الخلية دي بتوريك ملخص سريع محليًا لملف `duplicate_groups.json` اللي اترفع.

In [ ]:
import json, os

fname = "dedupe_reports/duplicate_groups.json"
if not os.path.exists(fname):
    print(f"{fname}: مش موجود")
else:
    groups = json.load(open(fname, encoding="utf-8"))
    total_members = sum(len(g["members"]) for g in groups)
    total_review = sum(1 for g in groups for m in g["members"] if m["need_review"])
    print(f"عدد المجموعات: {len(groups)}")
    print(f"إجمالي العناصر داخل مجموعات: {total_members}")
    print(f"منهم محتاجين مراجعة بشرية (need_review=true): {total_review}")